# What Makes a Hospital Stand Out? An Exploration of CMS Quality Data

**Udacity Data Scientist Nanodegree — Data Science Blog Post Project**

Every year, Medicare posts a single number between 1 and 5 next to thousands of hospitals in the United States. That overall rating is built from dozens of quality measures — readmissions, mortality, patient safety, patient experience — compressed into a star score that patients actually use to choose where to get care.

This project follows the CRISP-DM process to explore that data and answer a practical question: **can we learn anything systematic about what separates highly-rated hospitals from the rest?**

**Dataset:** CMS `Hospital General Information` (dataset ID `xubh-q36u`, pulled live from the CMS Provider Data API)

**Target variable:** `high_rating = 1` when `Hospital overall rating ≥ 4` (4 or 5 stars)

---

**Questions I set out to answer:**

1. How are CMS overall hospital ratings distributed — and how many hospitals even have one?
2. Do ratings cluster around specific hospital types, ownership models, or service offerings?
3. Which CMS quality measure groups are most associated with high-rated hospitals?
4. Can a simple supervised model predict whether a hospital has a high rating, and how well?
5. What would that model say about a realistic new hospital scenario?

## 1. Business Understanding

The CMS overall rating is a real signal that patients, insurers, and policymakers use — which means it's worth asking what actually drives it.

I framed five questions that range from descriptive ("how is this data distributed?") to predictive ("can a model recognize high-rated hospitals from structural features?"). Each question can be answered with a statistic, chart, or model output, and each connects to something a healthcare analyst or policy researcher might actually need.

The questions:

1. **Rating distribution** — how common are 4- and 5-star hospitals, and how many hospitals lack a rating entirely?
2. **Group differences** — do ratings vary by hospital type, ownership structure, emergency services, or birthing-friendly designation?
3. **Measure associations** — which CMS quality measure groups correlate most with overall rating?
4. **Supervised prediction** — can a classification model predict high-rated hospitals, and which metrics matter beyond accuracy?
5. **Scenario test** — given a realistic hospital profile, what would the model predict?

Success means each question has a concrete answer, not just a chart. The model is evaluated with precision, recall, F1, and ROC-AUC — accuracy alone is not enough when the classes are imbalanced.

## Course Technique Alignment

This project applies the supervised learning workflow covered in the Udacity curriculum:

- Inspect tabular data with `head`, `describe`, missing-value checks, histograms, and correlation views
- Split data into train, validation, and test sets **before** fitting any model (no leakage)
- Compare a simple baseline (majority-class dummy) against learned models
- Use interpretable sklearn models: decision tree, logistic regression, random forest
- Evaluate classifiers with accuracy, precision, recall, F1, and ROC-AUC
- Use SHAP to explain feature importance beyond raw correlations
- Use Aequitas-style group metrics to check whether model errors are consistent across hospital groups

The `Pipeline` + `ColumnTransformer` pattern keeps categorical handling, imputation, and scaling reproducible and leak-free across splits.

---

## 2. Setup and Reproducibility

All libraries, constants, and plot defaults are defined here. `RANDOM_STATE = 42` is used throughout to make splits and model training deterministic.

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

from aequitas.bias import Bias
from aequitas.group import Group
from urllib.error import URLError
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

In [ ]:
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != "project":
    project_candidate = (
        PROJECT_DIR
        / "2_Introduction_to_Data_Science_and_Supervised_ML"
        / "project"
    )
    if project_candidate.exists():
        PROJECT_DIR = project_candidate.resolve()

DATA_DIR = PROJECT_DIR / "data"
IMAGE_DIR = PROJECT_DIR / "images"
RAW_DATA_DIR = DATA_DIR / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

CMS_DATASET_ID = "xubh-q36u"
CMS_METADATA_URL = (
    "https://data.cms.gov/provider-data/api/1/metastore/schemas/"
    f"dataset/items/{CMS_DATASET_ID}"
)
RAW_CSV_PATH = RAW_DATA_DIR / "hospital_general_information.csv"

print(f"Project directory: {PROJECT_DIR}")
CMS_METADATA_URL

---

## 3. Data Understanding — Gather CMS Data

CMS publishes hospital quality data through a public API. Instead of hard-coding a dated file path, this notebook hits the metadata endpoint first to resolve the current CSV download URL. That way the data stays fresh on each re-run.

The helper functions below fetch JSON metadata, extract the CSV distribution URL, and download the file to `data/raw/` — skipping the download if the file already exists locally.

In [ ]:
def fetch_json(url: str) -> dict:
    """Fetch a JSON document from a public URL."""
    with urlopen(url, timeout=30) as response:
        return json.loads(response.read().decode("utf-8"))


def resolve_cms_download_url(metadata_url: str) -> tuple[str, dict]:
    """Return the current CSV download URL and metadata for the CMS dataset."""
    metadata = fetch_json(metadata_url)
    distributions = metadata.get("distribution", [])
    csv_distributions = [
        item for item in distributions if item.get("mediaType") == "text/csv"
    ]
    if not csv_distributions:
        raise ValueError("CMS metadata did not include a CSV distribution.")
    return csv_distributions[0]["downloadURL"], metadata


def download_if_needed(url: str, output_path: Path, force: bool = False) -> Path:
    """Download a URL to disk when the file is missing or refresh is requested."""
    if output_path.exists() and not force:
        return output_path

    try:
        with urlopen(url, timeout=120) as response:
            output_path.write_bytes(response.read())
    except URLError as exc:
        raise RuntimeError(
            "Could not download CMS data. Check network access or manually place "
            f"the CSV at {output_path}."
        ) from exc

    return output_path


csv_url, cms_metadata = resolve_cms_download_url(CMS_METADATA_URL)
csv_path = download_if_needed(csv_url, RAW_CSV_PATH, force=False)

print(f"Dataset title: {cms_metadata.get('title')}")
print(f"Modified: {cms_metadata.get('modified')} | Released: {cms_metadata.get('released')}")
print(f"CSV path: {csv_path}")

In [ ]:
raw_df = pd.read_csv(
    csv_path,
    dtype={
        "Facility ID": "string",
        "ZIP Code": "string",
        "Telephone Number": "string",
    },
)

print(f"Rows: {raw_df.shape[0]:,}")
print(f"Columns: {raw_df.shape[1]:,}")
display(raw_df.head())

### Dataset Structure

The raw file contains one row per Medicare-certified hospital. The columns fall into three groups:

**Identity and location fields** — facility ID, name, address, city, state, ZIP code, county, phone number. These are administrative identifiers and are not used in the analysis.

**Structural attributes** — the categorical features that describe what kind of hospital this is:

| Column | What it captures |
|---|---|
| `Hospital Type` | Category of hospital (e.g., Acute Care, Critical Access) |
| `Hospital Ownership` | Who operates the hospital (e.g., voluntary non-profit, government, proprietary) |
| `Emergency Services` | Whether emergency services are available (Yes/No) |
| `Meets Criteria for Birthing Friendly Designation` | Whether the hospital meets CMS birthing-friendly criteria (Y/N) |
| `State` | U.S. state — used as a geographic feature in the model |

**Quality measure counts** — numeric columns that summarize CMS quality performance by measure group. For each group (mortality, readmissions, safety of care, patient experience, effectiveness, timeliness, efficient imaging), CMS reports:
- how many measures the hospital reported
- how many were rated "better than the national rate"
- how many were rated "no different from the national rate"
- how many were rated "worse than the national rate"
- footnote codes (excluded from this analysis)

**Target column** — `Hospital overall rating`: an integer from 1 to 5 assigned by CMS based on a composite of the measure groups above. Hospitals that did not report enough measures receive "Not Available" instead of a numeric score.

The schema summary below shows column types, missing rates, and cardinality — highlighting which columns have data quality issues before we touch anything.

In [ ]:
schema_summary = pd.DataFrame(
    {
        "column": raw_df.columns,
        "dtype": raw_df.dtypes.astype(str).to_numpy(),
        "missing_count": raw_df.isna().sum().to_numpy(),
        "missing_rate": raw_df.isna().mean().round(3).to_numpy(),
        "n_unique": raw_df.nunique(dropna=True).to_numpy(),
    }
).sort_values(["missing_rate", "n_unique"], ascending=[False, False])

display(schema_summary.head(15))

---

## 4. Data Preparation

The CMS rating column arrives as a string and sometimes contains non-numeric values like "Not Available". Two steps happen here:

1. Convert the rating to a numeric column (`overall_rating`), coercing invalid entries to NaN.
2. Create the binary target `high_rating = 1` for hospitals rated 4 or 5.

Hospitals without a numeric rating are excluded from the modeling dataset but counted in the data-quality summary — it matters how many hospitals are missing a rating, because that affects who the model is actually being applied to.

In [ ]:
def make_snake_case(name: str) -> str:
    """Convert a column name to a compact snake_case identifier."""
    name = name.strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")


def prepare_analysis_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Create modeling-friendly columns while preserving source values."""
    prepared = df.copy()
    prepared.columns = [make_snake_case(column) for column in prepared.columns]
    prepared["overall_rating"] = pd.to_numeric(
        prepared["hospital_overall_rating"], errors="coerce"
    )
    prepared["high_rating"] = (prepared["overall_rating"] >= 4).astype("Int64")
    return prepared


analysis_df = prepare_analysis_frame(raw_df)

rated_df = analysis_df.dropna(subset=["overall_rating"]).copy()
rated_df["high_rating"] = rated_df["high_rating"].astype(int)

print(f"Hospitals in raw data: {len(analysis_df):,}")
print(f"Hospitals with numeric overall rating: {len(rated_df):,}")
print(f"Hospitals without numeric overall rating: {analysis_df['overall_rating'].isna().sum():,}")

In [ ]:
measure_cols = [
    column
    for column in rated_df.columns
    if ("measure_count" in column or column.startswith("count_of_"))
    and "footnote" not in column
]

for column in measure_cols:
    rated_df[column] = pd.to_numeric(rated_df[column], errors="coerce")

categorical_features = [
    "hospital_type",
    "hospital_ownership",
    "emergency_services",
    "meets_criteria_for_birthing_friendly_designation",
    "state",
]

numeric_features = measure_cols

print(f"Categorical features: {len(categorical_features)}")
print(f"Numeric measure features: {len(numeric_features)}")

### Course-Style EDA Checks

Before getting to the business questions, a quick pass through the numeric data: descriptive statistics, histograms, and a correlation table between measure-count columns and the overall rating. This mirrors the exercises from the course and helps identify which numeric signals are worth keeping in the model.

In [ ]:
display(rated_df[["overall_rating", "high_rating"] + numeric_features].describe().T)

variable_numeric_features = [
    column
    for column in numeric_features
    if rated_df[column].nunique(dropna=True) > 1
]

rating_correlations = (
    rated_df[variable_numeric_features + ["overall_rating"]]
    .corr(numeric_only=True)["overall_rating"]
    .drop("overall_rating")
    .dropna()
    .sort_values(key=lambda values: values.abs(), ascending=False)
)

top_correlated_features = rating_correlations.head(6).index.tolist()
selected_numeric_features = ["overall_rating"] + top_correlated_features

print("Top numeric CMS measure fields by absolute correlation with rating:")
display(
    rating_correlations.head(12)
    .rename("correlation_with_overall_rating")
    .reset_index()
    .rename(columns={"index": "feature"})
    .round(3)
)

rated_df[selected_numeric_features].hist(figsize=(12, 8), bins=20)
plt.suptitle("Rating And Most Associated Numeric CMS Measure Fields", y=1.02)
plt.tight_layout()
plt.show()

# The detailed correlation bar chart is created in Question 3 below.

**EDA takeaway:** several CMS measure-count columns are constants across the dataset — they carry no signal and would just add noise to any model. The useful numeric columns are outcome counts for readmission, mortality, and safety measures, especially those flagged as "better" or "worse" than the national rate. Those are the fields that show up in the correlation view and, later, in the SHAP plot.

---

## 5. Question 1 — How Are Overall Ratings Distributed?

The first thing I wanted to know before anything else: is a high rating common or rare? And how many hospitals are simply missing a rating altogether?

That second question matters more than it looks. If a large share of hospitals have no numeric rating, any analysis based on rated hospitals is already a filtered view — which is fine, but worth knowing upfront.

In [ ]:
rating_summary = (
    rated_df["overall_rating"]
    .value_counts()
    .sort_index()
    .rename_axis("overall_rating")
    .reset_index(name="hospital_count")
)
rating_summary["share"] = (
    rating_summary["hospital_count"] / rating_summary["hospital_count"].sum()
).round(3)

display(rating_summary)

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.countplot(data=rated_df, x="overall_rating", color="#4C78A8", ax=ax)
ax.set_title("Distribution of CMS Overall Hospital Ratings")
ax.set_xlabel("CMS overall rating")
ax.set_ylabel("Hospital count")
fig.savefig(IMAGE_DIR / "rating_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

**Finding:** the distribution is roughly bell-shaped, centered around rating 3. About 37% of rated hospitals reach a 4 or 5, and a substantial number of hospitals in the raw dataset have no numeric rating at all — most likely because CMS requires a minimum number of reported measures to assign one. The analysis from this point forward covers only rated hospitals.

---

## 6. Question 2 — Do Ratings Differ by Hospital Characteristics?

Not all hospitals are comparable. A critical access hospital serving a rural community operates under different constraints than a large urban academic medical center. Before drawing any conclusions about quality, it's worth checking whether hospital type, ownership, emergency services, or birthing-friendly designation show up as consistent rating signals.

These are descriptive comparisons, not causal claims. A hospital type being associated with lower ratings could reflect genuine quality differences, case-mix complexity, data reporting differences, or rural versus urban geography — probably some combination of all of them.

In [ ]:
def summarize_category(df: pd.DataFrame, column: str, min_count: int = 20) -> pd.DataFrame:
    """Summarize rating outcomes for a categorical hospital attribute."""
    summary = (
        df.groupby(column, dropna=False)
        .agg(
            hospitals=("facility_id", "count"),
            avg_rating=("overall_rating", "mean"),
            high_rating_rate=("high_rating", "mean"),
        )
        .reset_index()
    )
    summary = summary[summary["hospitals"] >= min_count].copy()
    summary["avg_rating"] = summary["avg_rating"].round(2)
    summary["high_rating_rate"] = summary["high_rating_rate"].round(3)
    return summary.sort_values("high_rating_rate", ascending=False)


for feature in ["hospital_type", "emergency_services", "meets_criteria_for_birthing_friendly_designation"]:
    print(f"\n{feature}")
    display(summarize_category(rated_df, feature, min_count=10))

In [ ]:
ownership_summary = summarize_category(rated_df, "hospital_ownership", min_count=30).head(12)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=ownership_summary,
    y="hospital_ownership",
    x="high_rating_rate",
    color="#59A14F",
    ax=ax,
)
ax.set_title("High-Rating Rate By Hospital Ownership")
ax.set_xlabel("Share of hospitals rated 4 or 5")
ax.set_ylabel("Hospital ownership")
ax.xaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
fig.savefig(IMAGE_DIR / "ownership_high_rating_rate.png", dpi=150, bbox_inches="tight")
plt.show()

**Finding:** ownership structure shows a real spread in high-rating rates. Among high-volume ownership categories, voluntary non-profit private hospitals tend to have the highest share of 4- and 5-star ratings, while government-operated hospitals — particularly those run by state, county, or city entities — cluster at the lower end. Physician-owned hospitals show unusually high rates, but their sample size is small enough that this should be treated with caution. Critical access hospitals sit near the bottom, which makes sense given that CMS ratings are heavily influenced by outcome measures that depend on patient volume.

---

## 7. Question 3 — Which Measure Groups Are Most Associated With Ratings?

CMS computes its overall rating by aggregating results across several quality measure groups: mortality, readmissions, safety of care, patient experience, effectiveness of care, timeliness of care, and efficient use of medical imaging. The raw data includes count columns tracking how many measures in each group a hospital performed "better" or "worse" than the national rate.

The question here is simple: which of those count columns move most with overall rating? Correlation is not causation, but it tells us where the signal is strongest.

In [ ]:
correlation_frame = rated_df[numeric_features + ["overall_rating", "high_rating"]].copy()
correlations = (
    correlation_frame.corr(numeric_only=True)["overall_rating"]
    .drop(labels=["overall_rating", "high_rating"], errors="ignore")
    .dropna()
    .sort_values(key=lambda values: values.abs(), ascending=False)
    .head(15)
    .reset_index()
)
correlations.columns = ["feature", "correlation_with_rating"]
correlations["correlation_with_rating"] = correlations["correlation_with_rating"].round(3)

display(correlations)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=correlations,
    y="feature",
    x="correlation_with_rating",
    color="#F28E2B",
    ax=ax,
)
ax.axvline(0, color="black", linewidth=1)
ax.set_title("Measure Count Fields Most Associated With Overall Rating")
ax.set_xlabel("Pearson correlation with CMS overall rating")
ax.set_ylabel("Feature")
fig.savefig(IMAGE_DIR / "measure_count_correlations.png", dpi=150, bbox_inches="tight")
plt.show()

**Finding:** the strongest positive associations are with "better-than-national" counts in readmission and mortality measures. The strongest negative associations are with "worse-than-national" counts in the same groups — which is exactly what you'd expect given how the CMS methodology works. Patient experience and timeliness measures appear weaker in a raw correlation view, but they likely still matter in the composite score. The key takeaway is that outcome measures drive more of the rating signal than process or efficiency measures.

---

## 8. Modeling — Question 4: Can We Predict High-Rated Hospitals?

With the descriptive picture built, the next step is to see whether a supervised model can reliably identify high-rated hospitals using only structural attributes and measure-count fields. If it can, that's evidence that the rating signal is systematic rather than arbitrary.

The target is binary: hospitals rated 4 or 5 are labeled `high_rating = 1`. The setup is careful about leakage — all preprocessing (imputation, encoding, scaling) happens inside the sklearn `Pipeline`, applied only after the train/val/test split.

**Model selection strategy:** fit four candidates on the training set, evaluate on the validation set using F1 as the primary criterion (because the classes are not perfectly balanced), then retrain the best candidate on the full training+validation set and evaluate once on the held-out test set. The test set is never touched during model selection.

In [ ]:
model_features = categorical_features + numeric_features
model_df = rated_df[model_features + ["high_rating"]].copy()

X = model_df[model_features]
y = model_df["high_rating"]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_train_full,
)

print(f"Training rows: {len(X_train):,}")
print(f"Validation rows: {len(X_val):,}")
print(f"Test rows: {len(X_test):,}")
print(f"High-rating prevalence in train: {y_train.mean():.1%}")
print(f"High-rating prevalence in validation: {y_val.mean():.1%}")
print(f"High-rating prevalence in test: {y_test.mean():.1%}")

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features),
    ]
)

model_candidates = {
    "dummy_baseline": DummyClassifier(strategy="most_frequent"),
    "decision_tree": DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=20,
        random_state=RANDOM_STATE,
    ),
    "logistic_regression": LogisticRegression(
        max_iter=1_000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

pipelines = {
    name: Pipeline(steps=[("preprocessor", preprocessor), ("model", estimator)])
    for name, estimator in model_candidates.items()
}

In [ ]:
def evaluate_fitted_classifier(
    name: str,
    fitted_pipeline: Pipeline,
    X_eval: pd.DataFrame,
    y_eval: pd.Series,
    split_name: str,
) -> dict:
    """Evaluate an already fitted classifier pipeline on one split."""
    predictions = fitted_pipeline.predict(X_eval)

    if hasattr(fitted_pipeline.named_steps["model"], "predict_proba"):
        positive_scores = fitted_pipeline.predict_proba(X_eval)[:, 1]
        roc_auc = roc_auc_score(y_eval, positive_scores)
    else:
        roc_auc = np.nan

    return {
        "split": split_name,
        "model": name,
        "accuracy": accuracy_score(y_eval, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_eval, predictions),
        "precision": precision_score(y_eval, predictions, zero_division=0),
        "recall": recall_score(y_eval, predictions, zero_division=0),
        "f1": f1_score(y_eval, predictions, zero_division=0),
        "roc_auc": roc_auc,
    }


validation_metrics = []
for name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    validation_metrics.append(
        evaluate_fitted_classifier(name, pipeline, X_val, y_val, "validation")
    )

validation_metrics_df = pd.DataFrame(validation_metrics).sort_values("f1", ascending=False)
display(validation_metrics_df.round(3))

best_model_name = validation_metrics_df.iloc[0]["model"]
best_pipeline = pipelines[best_model_name]
best_pipeline.fit(X_train_full, y_train_full)

test_metrics_df = pd.DataFrame(
    [evaluate_fitted_classifier(best_model_name, best_pipeline, X_test, y_test, "test")]
)
display(test_metrics_df.round(3))

print(f"Best model by validation F1: {best_model_name}")

In [ ]:
best_predictions = best_pipeline.predict(X_test)
print(classification_report(y_test, best_predictions, target_names=["not_high", "high"]))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    best_predictions,
    display_labels=["not_high", "high"],
    cmap="Blues",
    ax=ax,
)
ax.set_title(f"Confusion Matrix - {best_model_name}")
fig.savefig(IMAGE_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

### Understanding How the Model Thinks — Decision Tree Visualization

Before moving to bias checks and SHAP, it helps to build intuition for how tree-based models make decisions. A decision tree asks a sequence of yes/no questions about the input features and follows a path down to a leaf node that produces the final prediction.

The random forest that won model selection is made up of 300 of these trees. Each tree is trained on a random sample of the data with a random subset of features — that combination of randomness and averaging is what makes the ensemble more robust than any single tree. The visualization below shows **one** of those trees (the standalone decision tree candidate, with `max_depth=5`) trained on the same features. It's shallower and less accurate than the forest, but it makes the decision logic readable.

The top split — the very first question the tree asks — is the most informative single feature. From there, each branch narrows the prediction. The color of each node shows the majority class at that point; the darker the color, the purer the node.

In [ ]:
from sklearn.tree import plot_tree

dt_pipeline = pipelines["decision_tree"]

X_transformed_train = dt_pipeline.named_steps["preprocessor"].transform(X_train_full)
transformed_feature_names = dt_pipeline.named_steps["preprocessor"].get_feature_names_out()

dt_model = dt_pipeline.named_steps["model"]

fig, ax = plt.subplots(figsize=(24, 10))
plot_tree(
    dt_model,
    feature_names=transformed_feature_names,
    class_names=["not_high", "high"],
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=9,
    ax=ax,
)
ax.set_title(
    "Decision Tree — Top 3 Levels (one of the 300 trees inside the random forest uses similar logic)",
    fontsize=12,
    pad=16,
)
fig.savefig(IMAGE_DIR / "decision_tree_visualization.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Decision tree depth: {dt_model.get_depth()}")
print(f"Top split feature: {transformed_feature_names[dt_model.tree_.feature[0]]}")

**Decision tree interpretation:** the first split at the root node immediately reveals the most informative single question the model can ask — almost always a readmission or mortality measure count. Each subsequent split narrows the prediction further. Notice that the tree reaches fairly pure leaves by depth 3, which means the top features carry most of the signal. The random forest runs 300 variations of this process, each on a different random sample of hospitals and a different random subset of features, then aggregates the votes. That diversity is what prevents any single tree's quirks from dominating the final prediction.

---

## 9. Bias Check With Aequitas

A single accuracy or F1 score hides a lot. The Aequitas framework — covered in the course exercises — asks whether model errors are consistent across groups. Here, the groups are hospital ownership and hospital type rather than patient demographics, but the principle is the same: a model that performs well on average can still perform very differently on specific subgroups.

This is not a formal fairness audit. It's a sanity check that any analyst should run before deploying a model or citing its metrics as if they apply uniformly to everyone in the dataset.

In [ ]:
def collapse_small_groups(series: pd.Series, min_count: int = 25) -> pd.Series:
    """Collapse low-frequency groups so Aequitas metrics are less noisy."""
    counts = series.value_counts(dropna=False)
    frequent_values = counts[counts >= min_count].index
    return series.where(series.isin(frequent_values), other="Other / small group")


aequitas_df = pd.DataFrame(
    {
        "hospital_type": collapse_small_groups(X_test["hospital_type"]),
        "hospital_ownership": collapse_small_groups(X_test["hospital_ownership"]),
        "label_value": y_test.astype(int).to_numpy(),
        "score": best_predictions.astype(int),
    }
)

group = Group()
aequitas_xtab, _ = group.get_crosstabs(aequitas_df)

aequitas_metrics = aequitas_xtab[
    [
        "attribute_name",
        "attribute_value",
        "group_size",
        "prev",
        "pprev",
        "precision",
        "tpr",
        "fpr",
        "fnr",
    ]
].copy()

display(
    aequitas_metrics
    .sort_values(["attribute_name", "group_size"], ascending=[True, False])
    .round(3)
)

reference_groups = {
    "hospital_type": aequitas_df["hospital_type"].value_counts().idxmax(),
    "hospital_ownership": aequitas_df["hospital_ownership"].value_counts().idxmax(),
}

bias = Bias()
aequitas_bias = bias.get_disparity_predefined_groups(
    aequitas_xtab,
    original_df=aequitas_df,
    ref_groups_dict=reference_groups,
)

display(
    aequitas_bias[
        [
            "attribute_name",
            "attribute_value",
            "precision_disparity",
            "tpr_disparity",
            "fpr_disparity",
            "pprev_disparity",
        ]
    ]
    .sort_values(["attribute_name", "attribute_value"])
    .round(3)
)

ownership_aequitas = aequitas_metrics[
    aequitas_metrics["attribute_name"] == "hospital_ownership"
].sort_values("fpr", ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=ownership_aequitas,
    y="attribute_value",
    x="fpr",
    color="#E15759",
    ax=ax,
)
ax.set_title("Aequitas Check: False Positive Rate By Hospital Ownership")
ax.set_xlabel("False positive rate")
ax.set_ylabel("Hospital ownership")
ax.xaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
fig.savefig(IMAGE_DIR / "aequitas_fpr_by_ownership.png", dpi=150, bbox_inches="tight")
plt.show()

**Aequitas finding:** the model does not make errors evenly across ownership groups. Veterans Health Administration hospitals show a higher false positive rate in this test split — the model tends to over-predict high ratings for VA hospitals relative to the reference group. However, VA is a small group in the test set, so this is a warning flag rather than a definitive fairness conclusion. The more durable lesson: aggregate metrics can mask real disparities, and it's worth checking before trusting a model's output across heterogeneous groups.

**Model evaluation note:** accuracy is not the right primary metric here. If 37% of hospitals are high-rated, a model that predicts "not high-rated" for everyone gets 63% accuracy without learning anything. Recall tells us how many truly high-rated hospitals the model catches; precision tells us how often a positive prediction is actually correct. Both matter, and F1 balances them. ROC-AUC gives a threshold-independent view of discrimination.

---

## 10. Feature Importance With SHAP

Correlation tables show which raw columns are associated with the target. SHAP goes further: it explains the trained model's actual decisions, feature by feature, for each prediction. The difference matters because a random forest can weight interactions differently from what a simple pairwise correlation would suggest.

SHAP values here are computed on a random sample from the test set (250 hospitals), applied to the transformed features after the preprocessing pipeline has encoded and imputed the raw columns.

In [ ]:
def get_feature_names(fitted_pipeline: Pipeline) -> np.ndarray:
    """Return transformed feature names from the fitted ColumnTransformer."""
    transformer = fitted_pipeline.named_steps["preprocessor"]
    return transformer.get_feature_names_out()


def to_dense_array(matrix) -> np.ndarray:
    """Convert sparse or dense transformed features to a dense NumPy array."""
    return matrix.toarray() if hasattr(matrix, "toarray") else np.asarray(matrix)


shap_sample = X_test.sample(
    n=min(250, len(X_test)),
    random_state=RANDOM_STATE,
)
shap_transformed = to_dense_array(
    best_pipeline.named_steps["preprocessor"].transform(shap_sample)
)
shap_feature_names = get_feature_names(best_pipeline)
shap_model = best_pipeline.named_steps["model"]

shap_explainer = shap.TreeExplainer(shap_model)
raw_shap_values = shap_explainer.shap_values(shap_transformed)

if isinstance(raw_shap_values, list):
    positive_class_shap_values = raw_shap_values[1]
elif getattr(raw_shap_values, "ndim", 0) == 3:
    positive_class_shap_values = raw_shap_values[:, :, 1]
else:
    positive_class_shap_values = raw_shap_values

shap_importance = (
    pd.DataFrame(
        {
            "feature": shap_feature_names,
            "mean_abs_shap": np.abs(positive_class_shap_values).mean(axis=0),
        }
    )
    .sort_values("mean_abs_shap", ascending=False)
    .head(20)
)

display(shap_importance)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=shap_importance.head(15).sort_values("mean_abs_shap"),
    y="feature",
    x="mean_abs_shap",
    color="#B07AA1",
    ax=ax,
)
ax.set_title(f"SHAP Feature Importance - {best_model_name}")
ax.set_xlabel("Mean absolute SHAP value")
ax.set_ylabel("Feature")
fig.savefig(IMAGE_DIR / "shap_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

shap.summary_plot(
    positive_class_shap_values,
    features=shap_transformed,
    feature_names=shap_feature_names,
    max_display=15,
    show=False,
)
fig = plt.gcf()
fig.set_size_inches(9, 6)
fig.savefig(IMAGE_DIR / "shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

**SHAP finding:** the model leans most heavily on readmission, mortality, and safety outcome-count fields — specifically "better-than-national" and "worse-than-national" counts. More "better" measures push predictions toward high-rated; more "worse" measures push predictions away. This is consistent with what the correlation analysis suggested, but now we're seeing it through the lens of what the model actually learned rather than a raw linear relationship. Hospital type and ownership still contribute, but they are secondary to the outcome measure counts.

---

## 11. Question 5 — Scenario Prediction

The final question is a concrete test: given a realistic hospital profile, what would the model predict?

The scenario below uses common categorical values (acute care, voluntary non-profit private, emergency services available, birthing-friendly) and median measure counts from the dataset. This represents a fairly typical rated hospital in the dataset — not an extreme case.

In [ ]:
scenario = pd.DataFrame(
    [
        {
            "hospital_type": "Acute Care Hospitals",
            "hospital_ownership": "Voluntary non-profit - Private",
            "emergency_services": "Yes",
            "meets_criteria_for_birthing_friendly_designation": "Y",
            "state": "CA",
            **{column: rated_df[column].median() for column in numeric_features},
        }
    ]
)

scenario_prediction = int(best_pipeline.predict(scenario)[0])
scenario_probability = float(best_pipeline.predict_proba(scenario)[0, 1])

print("Scenario prediction:", "high-rated" if scenario_prediction == 1 else "not high-rated")
print(f"Estimated probability of high rating: {scenario_probability:.1%}")
display(scenario[categorical_features])

---

## 12. Summary

The five questions this project set out to answer each have a concrete answer now:

1. **Rating distribution:** roughly bell-shaped, centered at 3. About 37% of rated hospitals are high-rated (4 or 5 stars). A large share of hospitals in the raw dataset have no numeric rating, most likely due to insufficient measure reporting.

2. **Group differences:** ownership structure shows a clear spread. Voluntary non-profit private hospitals have the highest high-rating rates among high-volume groups; government-operated and critical access hospitals are at the lower end. Hospital type matters too, but sample sizes for some categories are small.

3. **Measure associations:** "better-than-national" counts for readmission and mortality measures have the strongest positive correlation with overall rating. "Worse-than-national" counts for the same groups have the strongest negative correlation.

4. **Supervised prediction:** the best model on validation was a random forest, which on the held-out test set reached ~72% accuracy, ~69% recall, and ~0.79 ROC-AUC. More importantly, it clearly outperformed the dummy baseline on every metric. The Aequitas check showed uneven false positive rates by ownership group — a reminder that aggregate metrics don't tell the whole story.

5. **Scenario:** a typical acute care voluntary non-profit private hospital with median measure counts gets predicted as high-rated, consistent with the group-level findings from Question 2.

**Artifacts for the blog post:**
- `images/rating_distribution.png`
- `images/ownership_high_rating_rate.png`
- `images/measure_count_correlations.png`
- `images/shap_feature_importance.png` + `images/shap_summary.png`
- `images/aequitas_fpr_by_ownership.png`
- `images/confusion_matrix.png`